**Unbuntu and WSL complete setup**

Update Unbuntu

In [ ]:
sudo apt update
sudo apt upgrade -y

**Required System Packages**

In [ ]:
sudo apt install -y \
python3 \
python3-venv \
python3-pip \
build-essential \
petsc-dev \
libpetsc-real-dev \
libpetsc-real3.19-dev \
python3-petsc4py \
python3-petsc4py-real \
python3-petsc4py-real3.19

**Create JAX-FEM Environment** 

In [ ]:
python3 -m venv jaxfem
source jaxfem/bin/activate

**Verify**

In [ ]:
which python
python --version

**Expected:**

.../jaxfem/bin/python
Python 3.12.x

**Locate PETSc python package** 

In [ ]:
find /usr/lib -name petsc4py 2>/dev/null

**Typical Result** 

/usr/lib/petscdir/petsc3.19/x86_64-linux-gnu-real/lib/python3/dist-packages/petsc4py

**Configure PETSc path**

In [ ]:
export PYTHONPATH=/usr/lib/petscdir/petsc3.19/x86_64-linux-gnu-real/lib/python3/dist-packages:$PYTHONPATH

**Test** 

In [ ]:
python -c "import petsc4py; print(petsc4py.__file__)"

**Persist Permanently**

In [ ]:
echo 'export PYTHONPATH=/usr/lib/petscdir/petsc3.19/x86_64-linux-gnu-real/lib/python3/dist-packages:$PYTHONPATH' >> ~/.bashrc
source ~/.bashrc

**Install Numpy 1.26.4**

Must be specific version to be compatible with this PETSc

In [ ]:
pip install numpy==1.26.4

**Verify PETSc**

In [ ]:
python - <<'EOF'
from petsc4py import PETSc
import numpy as np

print("PETSc OK")
print("NumPy:", np.__version__)
EOF

**Install Known Good JAX Stack**

remove incompatable versions first

In [ ]:
python -m pip uninstall -y jax jaxlib

Updrade packaging tools

In [ ]:
python -m pip install --upgrade pip setuptools wheel

Install exact versions 

In [ ]:
python -m pip install \
jax==0.4.28 \
jaxlib==0.4.28

**Install FEM Dependencies**

In [ ]:
python -m pip install \
meshio \
gmsh \
scipy \
matplotlib \
pyfiglet

**Install correct Basix package**

Remove wrong package if present

In [ ]:
python -m pip uninstall -y basix

Install the correct one 

In [ ]:
python -m pip install fenics-basix==0.10.0

**Install JAX FEM**

In [ ]:
python -m pip install jax-fem==0.0.11

Or if running from the source

In [ ]:
git clone <your-jax-fem-repo>
cd jax-fem
python -m pip install -e .

**Verification**

Basix

In [ ]:
python - <<'EOF'
import basix

print("file:", basix.__file__)
print("version:", basix.__version__)
print("ElementFamily exists:", hasattr(basix, "ElementFamily"))
EOF

Expected: 

ElementFamily exists: True

**Verify versions**

In [ ]:
python - <<'EOF'
import numpy
import jax
import basix

print("numpy =", numpy.__version__)
print("jax =", jax.__version__)
print("basix =", basix.__version__)
EOF

Expected: 

numpy = 1.26.4
jax = 0.4.28
basix = 0.10.0

**Verify JAX-FEM**

In [ ]:
python -m pip show jax-fem

Expected: 

Version: 0.0.11

**Running your FEM Script**

Activate environment:

In [ ]:
source /jaxfem/bin/activate

Ensure PETSc path exists: 

In [ ]:
export PYTHONPATH=/usr/lib/petscdir/petsc3.19/x86_64-linux-gnu-real/lib/python3/dist-packages:$PYTHONPATH

Navigate to your project:

In [ ]:
cd ~/path/to/project

Run: 

In [ ]:
python Example.py

Or

In [ ]:
python my_fem_script.py

Expected output: 

[INFO] Solving a problem with ...

[DEBUG] JAX Solver - Finished solving

[INFO] max of dofs = ...

[INFO] min of dofs = ...

This warning is normal:

VTU requires 3D points, but 2D points given.

**If launching from windows (e.g going through with powershell)**

If using WSL Unbuntu: 

Open Unbuntu: 

In [ ]:
wsl

Activate environment:

In [ ]:
wsl bash -c "source ~/jaxfem/bin/activate && python --version"

Run FEM Script: 

In [ ]:
wsl bash -c "source ~/jaxfem/bin/activate && cd ~/path/to/project && python Example.py"

Run any script: 

In [ ]:
wsl bash -c "source ~/jaxfem/bin/activate && cd ~/path/to/project && python my_fem_script.py"

One command recovery: 

If JAX or Numpy become corrupted run 

In [ ]:
python -m pip uninstall -y jax jaxlib numpy basix
python -m pip install \
numpy==1.26.4 \
jax==0.4.28 \
jaxlib==0.4.28 \
fenics-basix==0.10.0

** Copy paste one and all command into powershell**

In [ ]:
$Desktop = [Environment]::GetFolderPath("Desktop")
if (-not (Test-Path $Desktop)) {
    $Desktop = "$env:USERPROFILE"
}

$Report = Join-Path $Desktop "JAXFEM_INSTALL_REPORT.txt"

"==========================================================
JAX-FEM INSTALL + DIAGNOSTIC REPORT
==========================================================" | Set-Content $Report

function Log {
    param([string]$Text)
    $line = "[$(Get-Date -Format HH:mm:ss)] $Text"
    Write-Host $line
    Add-Content -Path $Report -Value $line
}

Log "Starting WSL + install pipeline"

Log "WSL Status"
wsl --status 2>&1 | Tee-Object -FilePath $Report -Append
wsl -l -v 2>&1 | Tee-Object -FilePath $Report -Append


# -----------------------------
# WRITE WSL SCRIPT
# -----------------------------

$WSL_SCRIPT_PATH = "$env:TEMP\install_jaxfem.sh"

@'
#!/usr/bin/env bash
set -e

LOGFILE="/tmp/jaxfem_install.log"
exec > >(tee -a "$LOGFILE") 2>&1

PASS=0
FAIL=0

log_step () {
    echo
    echo "=========================================================="
    echo "$(date '+%Y-%m-%d %H:%M:%S') - $1"
    echo "=========================================================="
}

run_step () {
    STEP="$1"
    shift

    log_step "$STEP"

    "$@"
    CODE=$?

    if [ $CODE -eq 0 ]; then
        echo "[PASS] $STEP"
        PASS=$((PASS+1))
    else
        echo "[FAIL] $STEP (exit=$CODE)"
        FAIL=$((FAIL+1))
    fi

    return $CODE
}

echo "SYSTEM INFO"
uname -a
cat /etc/os-release

# -----------------------------
# IMPORTANT: NO -n HERE (WANTS PASSWORD PROMPT)
# -----------------------------

run_step "APT UPDATE" sudo apt update
run_step "APT UPGRADE" sudo apt upgrade -y

run_step "INSTALL SYSTEM PACKAGES" sudo apt install -y \
    python3 python3-venv python3-pip build-essential \
    petsc-dev libpetsc-real-dev libpetsc-real3.19-dev \
    python3-petsc4py python3-petsc4py-real python3-petsc4py-real3.19

echo
log_step "CREATE VENV"

python3 -m venv ~/jaxfem
source ~/jaxfem/bin/activate

echo "Python:"
which python
python --version

echo
log_step "TEST PETSC4PY"
python -c 'import petsc4py; print("petsc4py OK:", petsc4py.__file__)'

run_step "INSTALL NUMPY 1.26.4" pip install numpy==1.26.4

python - <<EOF
from petsc4py import PETSc
import numpy as np
print("PETSc OK")
print("NumPy:", np.__version__)
EOF

run_step "REMOVE JAX" python -m pip uninstall -y jax jaxlib || true
run_step "UPGRADE PIP TOOLING" python -m pip install --upgrade pip setuptools wheel
run_step "INSTALL JAX STACK" python -m pip install jax==0.4.28 jaxlib==0.4.28

run_step "FEM DEPENDENCIES" python -m pip install meshio gmsh scipy matplotlib pyfiglet

run_step "REMOVE BASIX" python -m pip uninstall -y basix || true
run_step "INSTALL FENICS BASIX" python -m pip install fenics-basix==0.10.0
run_step "INSTALL JAX FEM" python -m pip install jax-fem==0.0.11

echo
log_step "VERIFICATION"

python - <<EOF
mods = ["numpy","jax","jaxlib","basix","jax_fem","petsc4py","meshio","gmsh","scipy","matplotlib"]

for m in mods:
    print("\n====", m, "====")
    try:
        mod = __import__(m)
        print("SUCCESS")
        if hasattr(mod,"__version__"):
            print("version =", mod.__version__)
    except Exception as e:
        print("FAILED", repr(e))
EOF

echo
log_step "SUMMARY"
echo "PASS=$PASS"
echo "FAIL=$FAIL"

echo "LOG FILE: $LOGFILE"
'@ | Set-Content -Encoding UTF8 $WSL_SCRIPT_PATH


# -----------------------------
# RUN WITH FULL INTERACTIVE OUTPUT
# -----------------------------

$WSL_PATH = wsl wslpath $WSL_SCRIPT_PATH

Log "Starting WSL execution (interactive mode)"
Log "Script path: $WSL_PATH"

# CRITICAL: do NOT buffer output, run directly in tty-like mode
wsl bash "$WSL_PATH"

Log "Done"

Write-Host ""
Write-Host "=========================================================="
Write-Host "REPORT SAVED TO:"
Write-Host $Report
Write-Host "=========================================================="

**Quick persistant install process in C:Github/LowCostAgilityForge** 

Copy past power shell block should work when restarting computer potentially